## Data Transformation

In [1]:
import os
%pwd

'd:\\Data Science\\END to END Proj\\Introvert vs Extrovert\\Introvert-Vs-Extrovert\\research'

In [2]:
os.chdir("../")

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    train_data_path: Path
    org_combined_path: Path
    x_train_path: Path
    x_val_path: Path
    y_train_path: Path
    y_val_path: Path
    ordinal_encoder_path: Path
    label_encoder_path: Path


In [4]:
from src.IntrovertVsExtrovert.utils.common import read_yaml, create_directories
from src.IntrovertVsExtrovert.constant import *

class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])
        
    def get_data_transformation_config(self) -> DataTransformationConfig:
        cfg = self.config.data_transformation
        create_directories([cfg.root_dir])

        return DataTransformationConfig(
            root_dir=cfg.root_dir,
            train_data_path=cfg.train_data_path,
            org_combined_path=cfg.org_combined_path,
            x_train_path=cfg.x_train_path,
            x_val_path=cfg.x_val_path,
            y_train_path=cfg.y_train_path,
            y_val_path=cfg.y_val_path,
            ordinal_encoder_path=cfg.ordinal_encoder_path,
            label_encoder_path=cfg.label_encoder_path
        )

In [6]:
# personality_prediction/components/data_transformation.py
import pandas as pd
import joblib
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
from pathlib import Path

CAT_COLS = ["Stage_fear", "Drained_after_socializing", "P2"]

class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def _merge_datasets(self, train_df: pd.DataFrame, org_df: pd.DataFrame) -> pd.DataFrame:
        key_cols = [
            "Time_spent_Alone", "Stage_fear", "Social_event_attendance",
            "Going_outside", "Drained_after_socializing",
            "Friends_circle_size", "Post_frequency"
        ]
        org_df = org_df.rename(columns={"Personality": "P2"})
        org_df = org_df.drop_duplicates(subset=key_cols)
        merged = train_df.merge(org_df, how="left", on=key_cols)
        return merged

    def transform(self):
        # 1 Load validated, cleaned CSVs from Stage 2
        train_df = pd.read_csv(self.config.train_data_path)
        org_df   = pd.read_csv(self.config.org_combined_path)

        # 2 Drop id if still present (safety)
        if "id" in train_df.columns:
            train_df = train_df.drop(columns=["id"])

        # 3 Merge & add P2
        train_df = self._merge_datasets(train_df, org_df)

        # 4 Target encode (Introvert=0, Extrovert=1)
        le = LabelEncoder()
        train_df["Personality"] = le.fit_transform(train_df["Personality"])

        # 5 Feature/target split
        X = train_df.drop(columns=["Personality"])
        y = train_df["Personality"]

        # 6 Ordinal‑encode categorical features (Stage_fear, Drained_after_socializing, P2)
        ord_enc = OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )
        X[CAT_COLS] = ord_enc.fit_transform(X[CAT_COLS])

        # 7 Train/val split
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )

        # 8 Persist everything
        Path(self.config.root_dir).mkdir(parents=True, exist_ok=True)
        X_train.to_parquet(self.config.x_train_path, index=False)
        X_val.to_parquet(self.config.x_val_path, index=False)
        y_train.to_csv(self.config.y_train_path, index=False, header=False)
        y_val.to_csv(self.config.y_val_path, index=False, header=False)
        joblib.dump(ord_enc, self.config.ordinal_encoder_path)
        joblib.dump(le, self.config.label_encoder_path)

        print("✅ Data transformation complete.")
        return X_train, X_val, y_train, y_val


In [7]:
try:
    # 1️⃣  Load config
    config = ConfigurationManager()
    data_trans_cfg = config.get_data_transformation_config()

    # 2️⃣  Run transformation
    transformer = DataTransformation(config=data_trans_cfg)
    X_train, X_val, y_train, y_val = transformer.transform()

    print("✅ Data Transformation Successful.")
except Exception as e:
    print(f"❌ Exception during transformation: {e}")


[2025-07-12 11:41:04,976: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-07-12 11:41:04,987: INFO: common: yaml file: params.yaml loaded successfully]
[2025-07-12 11:41:04,992: INFO: common: yaml file: schema.yaml loaded successfully]
[2025-07-12 11:41:04,996: INFO: common: created directory at: artifacts]
[2025-07-12 11:41:05,004: INFO: common: created directory at: artifacts/data_transformation]
✅ Data transformation complete.
✅ Data Transformation Successful.
